# Uso de script de modularización

### Paso 1 — Carga del panel de ventanas electorales y Revisión de variables



In [1]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")  
import ml_models
from ml_models.cargar_panel import cargar_panel, columnas_candidatas 
import importlib
import ml_models.lasso
from ml_models.lasso import *
importlib.reload(ml_models.lasso)

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel,f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 12 filas x 151 columnas
provincial: 12 filas x 151 columnas
nacional: 7 filas x 151 columnas


In [2]:
# delta_voto_exit_total_pct se retiró de panel_ventanas.csv por ser
# exactamente redundante con sus dos componentes (D22) -- se reconstruye
# acá porque este notebook la usa como hipótesis propia (exit total, no
# sus partes por separado).
for nivel in NIVELES:
    paneles[nivel]["delta_voto_exit_total_pct"] = (
        paneles[nivel]["delta_voto_exit_ausentismo_pct"] + paneles[nivel]["delta_voto_exit_blanco_nulo_pct"]
    )

/tmp/ipykernel_111563/1328794335.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  paneles[nivel]["delta_voto_exit_total_pct"] = (


In [3]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES: 
    df = paneles[nivel]
    cols_vc = columnas_candidatas(df, excluir_adicional=["delta_voto_exit_total_pct"])

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc)} variables candidatas, N={len(df)}")
corr_por_nivel["municipal"] 

municipal: 117 variables candidatas, N=7
provincial: 117 variables candidatas, N=7
nacional: 117 variables candidatas, N=7


,desocupacion_cobertura_parcial,desocupacion_delta_nivel,desocupacion_delta_pendiente,desocupacion_final_vc,desocupacion_nivel_vc,desocupacion_nivel_vl,desocupacion_pendiente_vc,desocupacion_pendiente_vl,desocupacion_volatilidad_vc,desocupacion_volatilidad_vl,...,tc_oficial_cobertura_parcial,tc_oficial_delta_nivel,tc_oficial_delta_pendiente,tc_oficial_final_vc,tc_oficial_nivel_vc,tc_oficial_nivel_vl,tc_oficial_pendiente_vc,tc_oficial_pendiente_vl,tc_oficial_volatilidad_vc,tc_oficial_volatilidad_vl
desocupacion_cobertura_parcial,1.000000,0.059879,0.365129,0.507663,0.490893,0.165688,-0.341839,0.047982,0.278257,-0.039538,...,NaN,-0.272743,-0.299844,-0.317495,-0.308755,-0.294031,-0.327083,-0.277194,-0.325106,-0.280969
desocupacion_delta_nivel,0.059879,1.000000,-0.704051,-0.287664,-0.477285,-0.777436,0.644708,0.995133,-0.407640,-0.847830,...,NaN,0.205776,0.110870,0.180213,0.197853,0.193566,0.144599,0.199828,0.147865,0.192945
desocupacion_delta_pendiente,0.365129,-0.704051,1.000000,0.678540,0.609044,0.667657,-0.278913,-0.721506,0.411142,0.517413,...,NaN,-0.013043,0.001606,-0.024382,-0.025642,-0.034030,-0.016657,-0.014155,-0.016543,-0.014519
desocupacion_final_vc,0.507663,-0.287664,0.678540,1.000000,0.967016,0.735748,-0.892786,-0.324685,0.485517,0.400098,...,NaN,-0.225549,-0.315856,-0.219954,-0.209513,-0.257800,-0.233651,-0.233141,-0.231804,-0.240848
desocupacion_nivel_vc,0.490893,-0.477285,0.609044,0.967016,1.000000,0.899991,-0.919701,-0.524422,0.663285,0.617906,...,NaN,-0.280850,-0.364126,-0.266020,-0.255274,-0.299475,-0.281650,-0.286523,-0.279971,-0.292257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tc_oficial_nivel_vl,-0.294031,0.193566,-0.034030,-0.257800,-0.299475,-0.333563,0.261991,0.193741,-0.102140,-0.193025,...,NaN,0.995892,0.978780,0.999631,0.999409,1.000000,0.993037,0.997464,0.993922,0.998647
tc_oficial_pendiente_vc,-0.327083,0.144599,-0.016657,-0.233651,-0.281650,-0.336658,0.213888,0.151610,-0.164768,-0.154158,...,NaN,0.980991,0.995946,0.995588,0.989649,0.993037,1.000000,0.984806,0.999966,0.988452
tc_oficial_pendiente_vl,-0.277194,0.199828,-0.014155,-0.233141,-0.286523,-0.329545,0.268812,0.197468,-0.102463,-0.213937,...,NaN,0.999782,0.967473,0.996561,0.999316,0.997464,0.984806,1.000000,0.986196,0.999747
tc_oficial_volatilidad_vc,-0.325106,0.147865,-0.016543,-0.231804,-0.279971,-0.336782,0.213485,0.154449,-0.163047,-0.157353,...,NaN,0.982549,0.995297,0.996304,0.990772,0.993922,0.999966,0.986196,1.000000,0.989660


### Paso 2 — Sub-selección: colapsar clusters redundantes

La matriz de correlación mostró un cluster de colinealidad casi perfecta entre `ipc_*` y `tc_oficial_*` (r > 0.98 en todo el bloque), además de pares menores en `desocupacion`, `resultado_fiscal` y `salario_real`. Se busca simplificar reduciendo los datos que son redundantes para evitar que LASSO elija de forma aleatoria entre esas opciones.

**Parametrización fijada:**

| Decisión | Valor |
|---|---|
| Umbral de redundancia | `\|r\| ≥ 0.90` |
| Alcance | Transitivo (single-linkage): si A-B≥0.90 y B-C≥0.90, A/B/C van al mismo cluster aunque A-C no llegue al umbral |
| Desempate 1 | Sufijo `_nivel_vc` preferido sobre `_final`/`_pendiente`/`_volatilidad`/`_acum` |
| Desempate 2 | Prioridad teórica de la variable (`ipc` > `desocupacion` > `icg` > `icc` > `salario_real` > `tc_oficial` > `reservas` > `resultado_fiscal`) — orden de relevancia en la literatura de voto económico citada, no un criterio estadístico |

In [4]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas candidatas, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")



Nivel: municipal


De 117 columnas candidatas, quedan 69 tras colapsar clusters (umbral=0.9)

cluster (2): ['desocupacion_delta_nivel', 'desocupacion_pendiente_vl'] -> queda: desocupacion_delta_nivel
cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (4): ['desocupacion_nivel_vl', 'emae_final_vc', 'emae_nivel_vc', 'emae_nivel_vl'] -> queda: emae_nivel_vc
cluster (2): ['emae_delta_nivel', 'emae_pendiente_vl'] -> queda: emae_pendiente_vl
cluster (5): ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> queda: hacinamiento_medio_cobertura_parcial
cluster (2): ['icc_delta_nivel', 'icc_pendiente_vl'] -> queda: icc_delta_nivel
cluster (2): ['icg_cobertura_parcial', 'reservas_cobertura_parcial'] -> queda: icg_cobertura_parcial
cluster (2):

De 117 columnas candidatas, quedan 69 tras colapsar clusters (umbral=0.9)

cluster (2): ['desocupacion_delta_nivel', 'desocupacion_pendiente_vl'] -> queda: desocupacion_delta_nivel
cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (4): ['desocupacion_nivel_vl', 'emae_final_vc', 'emae_nivel_vc', 'emae_nivel_vl'] -> queda: emae_nivel_vc
cluster (2): ['emae_delta_nivel', 'emae_pendiente_vl'] -> queda: emae_pendiente_vl
cluster (5): ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> queda: hacinamiento_medio_cobertura_parcial
cluster (2): ['icc_delta_nivel', 'icc_pendiente_vl'] -> queda: icc_delta_nivel
cluster (2): ['icg_cobertura_parcial', 'reservas_cobertura_parcial'] -> queda: icg_cobertura_parcial
cluster (2):

De 117 columnas candidatas, quedan 33 tras colapsar clusters (umbral=0.9)

cluster (59): ['desocupacion_cobertura_parcial', 'desocupacion_pendiente_vc', 'hacinamiento_medio_cobertura_parcial', 'hacinamiento_medio_nivel_vc', 'icc_final_vc', 'icc_nivel_vc', 'icc_nivel_vl', 'icc_volatilidad_vc', 'icc_volatilidad_vl', 'ipc_acum_vl', 'ipc_cobertura_parcial', 'ipc_delta_nivel', 'ipc_delta_pendiente', 'ipc_final_vc', 'ipc_nivel_vc', 'ipc_nivel_vl', 'ipc_pendiente_vc', 'ipc_pendiente_vl', 'ipc_volatilidad_vc', 'ipc_volatilidad_vl', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_nivel_vc', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_prestamo_bancario_delta_nivel', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_hogares_vendio_pertenencias_delta_nivel', 'pct_sin_cobertura_salud_cobertura_parcial', 'pct_sin_cobertura_salud_nivel_vc', 'resultado_fiscal_cobertura_parcial', 'resultado_fiscal_delta_nivel', 'resultado_fiscal_fina

In [5]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_voto_exit_total_pct")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2003_2005']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'tc_oficial_cobertura_parcial']
municipal: N=10, P=66
[provincial] excluye 2 fila(s) por NaN: ['provincial_2001_2003', 'provincial_2003_2005']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'tc_oficial_cobertura_parcial']
provincial: N=10, P=66
[nacional] excluye 1 fila(s) por NaN: ['nacional_2011_2013']
[nacional] excluye columna(s) sin varianza: ['emae_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'reservas_cobertura_parcial', 'tc_oficial_cobertura_parcial']
nacional: N=6, P=28


In [6]:
faltantes = columnas_nan("nacional", "nacional_2013_2015", columnas_finales_por_nivel["nacional"], paneles)
print("Columna(s) que rompen nacional_2013_2015:", faltantes)

Columna(s) que rompen nacional_2013_2015: []


In [7]:
for nivel, id_t in [("municipal", "municipal_2001_2003"), ("provincial", "provincial_2001_2003")]:
    faltantes = columnas_nan(nivel, id_t, columnas_finales_por_nivel[nivel],paneles)
    print(f"{id_t}: NaN en -> {faltantes}")


municipal_2001_2003: NaN en -> ['desocupacion_delta_nivel', 'desocupacion_delta_pendiente', 'emae_nivel_vc', 'desocupacion_volatilidad_vl', 'emae_pendiente_vl', 'emae_delta_pendiente', 'emae_pendiente_vc', 'emae_volatilidad_vc', 'emae_volatilidad_vl', 'hacinamiento_medio_delta_nivel', 'hacinamiento_medio_nivel_vc', 'icc_delta_nivel', 'icc_delta_pendiente', 'icc_nivel_vl', 'icc_volatilidad_vl', 'icg_delta_nivel', 'icg_delta_pendiente', 'icg_nivel_vl', 'icg_volatilidad_vl', 'pct_hogares_ayuda_social_gobierno_delta_nivel', 'pct_hogares_prestamo_bancario_delta_nivel', 'pct_hogares_prestamo_bancario_nivel_vc', 'pct_hogares_vendio_pertenencias_nivel_vc', 'pct_sin_cobertura_salud_delta_nivel', 'pct_sin_cobertura_salud_nivel_vc', 'reservas_delta_nivel', 'reservas_delta_pendiente', 'reservas_nivel_vl', 'reservas_volatilidad_vl', 'resultado_fiscal_delta_nivel', 'resultado_fiscal_delta_pendiente', 'salario_real_delta_nivel', 'salario_real_delta_pendiente', 'tasa_informalidad_pendiente_vl', 'tasa_

### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X` estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no se penaliza.

`soft_threshold`: operador proximal de L1, da la selección de variables (coeficiente exactamente en cero si `|z| <= alpha`). `lasso_coordinate_descent`: actualiza una coordenada de `β` a la vez hasta que el cambio máximo entre iteraciones sea `< tol`.

In [8]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    # Chequeo 2: alpha=0 debe coincidir con OLS
    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

    residuo_manual = y_centrado - X_std @ beta_alpha_cero
    residuo_ols = y_centrado - X_std @ beta_ols

    print(f"{nivel} - Residuo manual (debería ser ~0 si el sistema es subdeterminado):", np.max(np.abs(residuo_manual)))
    print(f"{nivel} - Residuo OLS (debería ser ~0 también):", np.max(np.abs(residuo_ols)))

municipal - KKT: {'error_max_en_activos': np.float64(1.842250583194982e-07), 'exceso_max_en_inactivos': np.float64(-0.041309121241204116), 'n_activos': np.int64(5)}
municipal - Máxima diferencia vs. OLS (alpha=0): 1.4196691734120535
municipal - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 9.663332134479674e-08
municipal - Residuo OLS (debería ser ~0 también): 1.0658141036401503e-14
provincial - KKT: {'error_max_en_activos': np.float64(1.61264619880086e-07), 'exceso_max_en_inactivos': np.float64(-0.006821540376391155), 'n_activos': np.int64(5)}
provincial - Máxima diferencia vs. OLS (alpha=0): 1.916260509263497
provincial - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 1.082091621446013e-07
provincial - Residuo OLS (debería ser ~0 también): 1.582067810090848e-14
nacional - KKT: {'error_max_en_activos': np.float64(7.182295288510332e-07), 'exceso_max_en_inactivos': np.float64(-0.06247616613406404), 'n_activos': np.int64(2)}
nacional - Máxima diferenc

### Paso 4 — Grilla de alpha + LOO-CV manual

In [9]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: alpha_min=3.6092  alpha_1se=7.3036  (techo grilla=7.3036)


provincial: alpha_min=4.2486  alpha_1se=8.5975  (techo grilla=8.5975)


nacional: alpha_min=2.0193  alpha_1se=4.0862  (techo grilla=19.2658)


In [10]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

--- municipal ---


   factor_extension      techo  alpha_min  alpha_1se    mse_min  mse_en_techo
0                 1   2.434519   2.434519   2.434519  20.879426     20.879426
1                 3   7.303556   3.609208   7.303556  16.999253     16.999253
2                10  24.345186   3.382753  24.345186  16.999253     16.999253

--- provincial ---


   factor_extension      techo  alpha_min  alpha_1se    mse_min  mse_en_techo
0                 1   2.865821   2.865821   2.865821  23.300545     23.300545
1                 3   8.597464   4.248620   8.597464  21.243337     21.243337
2                10  28.658214   3.982046  28.658214  21.243337     21.243337

--- nacional ---


   factor_extension      techo  alpha_min  alpha_1se    mse_min  mse_en_techo
0                 1   6.421934   1.805699   4.207192  29.722791     60.434515
1                 3  19.265801   2.019274   4.086182  29.706673     60.899959
2                10  64.219337   1.892577   4.409613  29.327918     60.899959



### Paso 6 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás puntos -- piso de comparación. `mse_en_alpha`: mismo esquema de LOO que `lasso_loocv_manual`, pero para un alpha puntual (se recalcula, no se guarda en la función de CV).

**Resultado:**

| Nivel | Mejora de `alpha_min` sobre baseline | Mejora de `alpha_1se` |
|---|---|---|
| Municipal | +40.7% | +13.3% |
| Provincial | +14.0% | +0.0% |
| Nacional | +0.0% | +0.0% |

Coeficientes en `alpha_min`: `icg_pendiente_vc` sobrevive en municipal (6.41) y provincial (4.41), `reservas_pendiente_vc` solo en municipal (0.96, débil). En `alpha_1se`: solo `icg_pendiente_vc` en municipal (2.81), nada en provincial ni nacional.

In [11]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

            baseline_mse  mejora_alpha_min_%  mejora_alpha_1se_%
nivel                                                           
municipal      16.999253            0.000000            0.000000
provincial     21.243337            0.000000            0.000000
nacional       60.899959           51.220536           26.420738


In [12]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

Coeficientes distintos de cero (alpha_min):


,municipal,provincial,nacional
desocupacion_cobertura_parcial,0.0,0.0,NaN
desocupacion_delta_nivel,0.0,0.0,NaN
desocupacion_final_vc,NaN,NaN,0.000000
desocupacion_volatilidad_vc,0.0,0.0,NaN
emae_cobertura_parcial,0.0,0.0,NaN
emae_delta_nivel,NaN,NaN,0.000000
emae_delta_pendiente,0.0,0.0,NaN
emae_final_vc,NaN,NaN,0.000000
emae_nivel_vl,NaN,NaN,0.000000
emae_pendiente_vc,0.0,0.0,3.957480


### Chequeo de estabilidad (leave-one-transition-out) — los tres niveles

**Municipal (alpha_1se=7.71):** Signo estable (`icg_pendiente_vc` siempre positivo), magnitud sensible a `2009_2011` y `2011_2013` (caída a 0.48/0.78 vs. 2.3-3.7 en el resto). Composición casi estable (`reservas_pendiente_vc` se activa débilmente solo al sacar `2005_2007`).

**Provincial (alpha_min=5.45 -- alpha_1se no tuvo sobrevivientes):**
- Signo estable: `icg_pendiente_vc` positivo en las 12 corridas, nunca en cero.
- Magnitud más variable que en municipal: rango 1.41 (sacando `2009_2011`) a 5.55 (sacando `2005_2007`). **`2009_2011` vuelve a ser la ventana de mayor apalancamiento, igual que en municipal** -- coincidencia entre niveles que sugiere que esa transición puntual tiene un peso real en el vínculo confianza-voto.
- Composición menos estable que en municipal: sacar `2015_2017` activa `icc_pendiente_vc` (2.02) mientras `icg` cae a 2.18 -- acá sí cambia cuál variable "aporta", no solo cuánto. `reservas_pendiente_vc`/`reservas_volatilidad_vc` aparecen de forma esporádica y débil en varias otras corridas, sin patrón consistente -- lectura: ruido, no señal.

**Nacional (alpha_min=11.85):** ninguna variable sobrevive en ninguna de las 7 corridas -- resultado nulo robusto.

**Síntesis:** `icg_pendiente_vc` es el hallazgo más sólido del ejercicio de LASSO -- signo positivo y consistente en municipal y provincial, con la particularidad de que la transición `2009_2011` reduce su magnitud en ambos niveles simultáneamente. Provincial es menos estable en composición que municipal (coherente con su menor mejora sobre baseline, 14% vs. 40.7%). Nacional no tiene ninguna señal individual bajo ningún criterio.

In [13]:
ALPHA_PARA_ESTABILIDAD = {
    "municipal": ("alpha_1se", resultados_cv["municipal"]["alpha_1se"]),
    "provincial": ("alpha_1se", resultados_cv["provincial"]["alpha_1se"]),
    "nacional": ("alpha_1se", resultados_cv["nacional"]["alpha_1se"]),
}

for nivel in NIVELES:
    df = paneles[nivel]
    criterio, alpha = ALPHA_PARA_ESTABILIDAD[nivel]
    X_df, y_ser = datos_final[nivel] 
    resultado = estabilidad_seleccion(nivel, alpha, df, columnas_finales_por_nivel[nivel], "delta_voto_exit_total_pct", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha={alpha:.3f}, criterio={criterio}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()

--- municipal (alpha=7.304, criterio=alpha_1se) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- provincial (alpha=8.597, criterio=alpha_1se) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- nacional (alpha=4.086, criterio=alpha_1se) ---
                        emae_pendiente_vc  icg_volatilidad_vc  \
sin_nacional_2013_2015           2.975798            0.000000   
sin_nacional_2015_2017           2.825823            0.000000   
sin_nacional_2017_2019           0.026269            0.108665   
sin_nacional_2019_2021           2.021816            0.000000   
sin_nacional_2021_2023           2.228297            0.000000   
sin_nacional_2023_2025           2.302869            0.000000   

                        salario_real_delta_pendiente  
sin_nacional_2013_2015                      0.000000  
sin_nacional_2015_2017                      0.000000  
sin_nacional_2017_2019                      0.000000  
sin_na